# Multivariate Wave Prediction Model (7 Variables, 3-Hour Forecast)

This model predicts 7 ocean wave parameters for a forecast horizon of **3 hours** using a lookback window of **48 hours**.
It incorporates **Static Features** (Bathymetry Depth & Land/Sea Mask) alongside the dynamic wave data.

In [ ]:
# ==============================================================================
# MASTER CONFIGURATION CELL
# ==============================================================================
import torch
import os

# --- Core Parameters ---
LOOKBACK_HOURS = 48
FORECAST_HORIZON_HOURS = 3

# --- File Paths (Updated) ---
RAW_DATA_PATH_1 = r'/home/aidl/Wave-Prediction/V2/new_dAtA/till xl.nc'   # Contains: ['VHM0_SW1', 'VHM0_WW']
RAW_DATA_PATH_2 = r'/home/aidl/Wave-Prediction/V2/new_dAtA/after_xl.nc' # Contains: ['VTM01_WW', 'VTM02', 'VTM01_SW1', 'VTM10', 'VSDX', 'VSDY', 'VSDmag']
STATIC_DATA_PATH = r'/home/aidl/Wave-Prediction/V2/new_dAtA/cmems_GEBCO_resampled_new.nc'

# Determine where to save processed data
BASE_DIR = os.path.dirname(RAW_DATA_PATH_1) if os.path.exists(os.path.dirname(RAW_DATA_PATH_1)) else '.'
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, f'processed_data_multivar_lb{LOOKBACK_HOURS}_fh{FORECAST_HORIZON_HOURS}')
MODEL_SAVE_PATH = os.path.join('models', f'convlstm_multivar_lb{LOOKBACK_HOURS}_fh{FORECAST_HORIZON_HOURS}.pth')
os.makedirs('models', exist_ok=True)

# --- Feature Engineering ---
# These are the 7 TARGET variables we want to predict
VARS_TO_USE = ['VHM0_SW1', 'VSDmag', 'VHM0_WW', 'VTM01_WW', 'VTM02', 'VTM01_SW1', 'VTM10']
NUM_TARGET_VARS = len(VARS_TO_USE)

# Input Channels = 7 Dynamic Variables + 2 Static Features (Depth, Mask)
INPUT_CHANNELS = len(VARS_TO_USE) + 2 

# --- Training Hyperparameters ---
LEARNING_RATE = 1e-4
BATCH_SIZE = 8
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10

# --- System Configuration ---
NUM_WORKERS = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("--- Configuration Summary ---")
print(f"Lookback: {LOOKBACK_HOURS} hrs | Forecast: {FORECAST_HORIZON_HOURS} hrs")
print(f"Variables ({NUM_TARGET_VARS}): {VARS_TO_USE}")
print(f"Input Channels: {INPUT_CHANNELS}")
print(f"Data Path 1: {RAW_DATA_PATH_1}")
print(f"Data Path 2: {RAW_DATA_PATH_2}")
print(f"Static Path: {STATIC_DATA_PATH}")
print(f"Device: {device}")

# STEP 1: PRE-PROCESSING SCRIPT

In [ ]:
import os
import pickle
import xarray as xr
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm
import warnings
import shutil

warnings.filterwarnings('ignore')

if os.path.exists(PROCESSED_DATA_DIR):
    print(f"Removing old processed data directory: {PROCESSED_DATA_DIR}")
    shutil.rmtree(PROCESSED_DATA_DIR)

print("\n--- Starting Multivariate Pre-processing ---")

# 1. Load and Merge Data
try:
    print("Loading Data Part 1...")
    ds1 = xr.open_dataset(RAW_DATA_PATH_1)
    print("Loading Data Part 2...")
    ds2 = xr.open_dataset(RAW_DATA_PATH_2)
    
    # Merge datasets based on time/coords
    print("Merging datasets...")
    ds_raw = xr.merge([ds1, ds2])
except FileNotFoundError:
    print(f"❌ ERROR: Files not found at {RAW_DATA_PATH_1} or {RAW_DATA_PATH_2}")
    raise

# Calculate Magnitude if needed
if 'VSDmag' in VARS_TO_USE and 'VSDmag' not in ds_raw:
    if 'VSDX' in ds_raw and 'VSDY' in ds_raw:
        ds_raw['VSDmag'] = np.sqrt(ds_raw['VSDX']**2 + ds_raw['VSDY']**2)
        print("Calculated VSDmag from VSDX and VSDY.")
    else:
        print("⚠️ Warning: VSDmag requested but VSDX/VSDY not found in merged dataset.")

ds_clean = ds_raw[VARS_TO_USE].astype(np.float32).fillna(0)
print("✅ Time-varying data loaded and merged.")

# 2. Load Static Data
try:
    ds_static = xr.open_dataset(STATIC_DATA_PATH)
    # Adapt variable names if GEBCO file differs from original
    if 'elevation' in ds_static:
        depth_raw = ds_static['elevation'].values
        ocean_mask = (depth_raw < 0).astype(np.float32)
    elif 'deptho' in ds_static:
        depth_raw = ds_static['deptho'].values
        ocean_mask = ds_static['mask'].values.astype(np.float32)
    else:
        print("⚠️ Static vars (elevation/deptho) not found in static file. Using zeros.")
        # Use dimensions from dynamic data if static load fails structure check
        lat = ds_clean.sizes['latitude']
        lon = ds_clean.sizes['longitude']
        depth_raw = np.zeros((lat, lon))
        ocean_mask = np.ones_like(depth_raw)

    ocean_depth = np.nan_to_num(depth_raw, nan=0.0).astype(np.float32)
except Exception as e:
    print(f"⚠️ Static data issue: {e}. Using dummy static features.")
    lat = ds_clean.sizes['latitude']
    lon = ds_clean.sizes['longitude']
    depth_raw = np.zeros((lat, lon))
    ocean_mask = np.ones_like(depth_raw)
    ocean_depth = depth_raw.astype(np.float32)

print("✅ Static data loaded.")

# 3. Splits
ds_train = ds_clean.sel(time=slice('2020-01-01', '2022-12-31'))
ds_val = ds_clean.sel(time=slice('2023-01-01', '2023-06-30'))
ds_test = ds_clean.sel(time=slice('2023-07-01', '2023-12-30'))
ds_splits = {'train': ds_train, 'val': ds_val, 'test': ds_test}

# 4. Fit Scalers (Train Only)
scalers = {}
for var in tqdm(VARS_TO_USE, desc="Fitting Scalers"):
    data = ds_train[var].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    scaler.fit(data)
    scalers[var] = scaler

depth_scaler = MinMaxScaler()
depth_scaler.fit(ocean_depth.reshape(-1, 1))
scalers['ocean_depth'] = depth_scaler

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
with open(os.path.join(PROCESSED_DATA_DIR, 'scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)

# 5. Prepare Static Tensor
scaled_depth = scalers['ocean_depth'].transform(ocean_depth.reshape(-1, 1)).reshape(ocean_depth.shape)
static_features_np = np.stack([scaled_depth, ocean_mask], axis=0) # Shape: [2, H, W]

# 6. Generate Sequences
total_window = LOOKBACK_HOURS + FORECAST_HORIZON_HOURS

for split_name, ds_split in ds_splits.items():
    split_dir = os.path.join(PROCESSED_DATA_DIR, split_name)
    os.makedirs(split_dir, exist_ok=True)
    
    # Pre-scale the entire split to speed up slicing
    scaled_vars_list = []
    for var in VARS_TO_USE:
        d = ds_split[var].values
        s = scalers[var].transform(d.reshape(-1, 1)).reshape(d.shape)
        scaled_vars_list.append(s)
    # Shape: [Time, Vars, H, W]
    split_data_np = np.stack(scaled_vars_list, axis=1) 
    
    num_seq = len(ds_split['time']) - total_window
    
    for i in tqdm(range(num_seq), desc=f"Saving {split_name}"):
        # Input: [Lookback, Vars, H, W]
        X_dynamic = split_data_np[i : i+LOOKBACK_HOURS]
        
        # Target: [Forecast, Vars, H, W]  <-- MULTIVARIATE TARGET
        y_target = split_data_np[i+LOOKBACK_HOURS : i+total_window]
        
        # Tile Static: [Lookback, 2, H, W]
        static_tiled = np.tile(static_features_np[np.newaxis, ...], (LOOKBACK_HOURS, 1, 1, 1))
        
        # Concat Input: [Lookback, Vars+2, H, W]
        X_final = np.concatenate([X_dynamic, static_tiled], axis=1)
        
        torch.save((torch.from_numpy(X_final.astype(np.float32)), 
                    torch.from_numpy(y_target.astype(np.float32))), 
                   os.path.join(split_dir, f'seq_{i:06d}.pt'))

print("✅ Pre-processing complete.")

# STEP 2: MODEL & TRAINING

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import glob
import time

class WaveDataset(Dataset):
    def __init__(self, directory):
        self.files = sorted(glob.glob(os.path.join(directory, '*.pt')))
    def __len__(self): return len(self.files)
    def __getitem__(self, idx): return torch.load(self.files[idx])

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2
        self.bias = bias
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim, kernel_size, padding=self.padding, bias=bias)

    def forward(self, x, h_c):
        h, c = h_c
        combined = torch.cat([x, h], dim=1)
        cc = self.conv(combined)
        i, f, o, g = torch.split(cc, self.hidden_dim, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, b, image_size, device):
        h, w = image_size
        return (torch.zeros(b, self.hidden_dim, h, w, device=device),
                torch.zeros(b, self.hidden_dim, h, w, device=device))

class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, batch_first=True, bias=True):
        super(ConvLSTM, self).__init__()
        self.num_layers = num_layers
        hidden_dims = [hidden_dim] * num_layers if isinstance(hidden_dim, int) else hidden_dim
        self.cell_list = nn.ModuleList()
        for i in range(num_layers):
            cur_input = input_dim if i == 0 else hidden_dims[i-1]
            self.cell_list.append(ConvLSTMCell(cur_input, hidden_dims[i], kernel_size, bias))

    def forward(self, x):
        # x: [Batch, Time, Channel, H, W]
        b, t, _, h, w = x.size()
        hidden_state = [cell.init_hidden(b, (h, w), x.device) for cell in self.cell_list]
        cur_input = x
        
        for layer_idx in range(self.num_layers):
            h, c = hidden_state[layer_idx]
            output_inner = []
            for time_step in range(t):
                h, c = self.cell_list[layer_idx](cur_input[:, time_step, :, :, :], (h, c))
                output_inner.append(h)
            # Stack outputs: [Batch, Time, Hidden, H, W]
            cur_input = torch.stack(output_inner, dim=1)
            hidden_state[layer_idx] = (h, c)
            
        return cur_input, hidden_state

class ConvLSTMNetMulti(nn.Module):
    def __init__(self, input_dim, num_target_vars, forecast_horizon, hidden_dims=[64, 32], kernel_size=(3,3)):
        super(ConvLSTMNetMulti, self).__init__()
        self.forecast_horizon = forecast_horizon
        self.num_target_vars = num_target_vars
        
        self.cl1 = ConvLSTM(input_dim, hidden_dims[0], kernel_size, 1)
        self.cl2 = ConvLSTM(hidden_dims[0], hidden_dims[1], kernel_size, 1)
        
        # Output Channels = Forecast Horizon * Num Variables
        # We map the last hidden state to ALL future steps x ALL variables at once
        self.out_channels = forecast_horizon * num_target_vars
        self.final_conv = nn.Conv2d(hidden_dims[1], self.out_channels, kernel_size=(1,1))

    def forward(self, x):
        # x: [Batch, Lookback, InputChannels, H, W]
        l1_out, _ = self.cl1(x)
        l2_out, _ = self.cl2(l1_out)
        
        # Use only the LAST time step's hidden state for prediction
        last_hidden = l2_out[:, -1, :, :, :]
        
        # Prediction: [Batch, (Forecast*Vars), H, W]
        pred_flat = self.final_conv(last_hidden)
        
        # Reshape to [Batch, Forecast, Vars, H, W]
        b, _, h, w = pred_flat.size()
        pred_reshaped = pred_flat.view(b, self.forecast_horizon, self.num_target_vars, h, w)
        
        return pred_reshaped

def train_model():
    train_ds = WaveDataset(os.path.join(PROCESSED_DATA_DIR, 'train'))
    val_ds = WaveDataset(os.path.join(PROCESSED_DATA_DIR, 'val'))
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    model = ConvLSTMNetMulti(input_dim=INPUT_CHANNELS, 
                             num_target_vars=NUM_TARGET_VARS,
                             forecast_horizon=FORECAST_HORIZON_HOURS).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler()
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    print("\n--- Starting Training ---")
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                pred = model(X)
                loss = criterion(pred, y)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            
        avg_train = train_loss / len(train_loader)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                with torch.cuda.amp.autocast():
                    pred = model(X)
                    loss = criterion(pred, y)
                val_loss += loss.item()
        avg_val = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}: Train Loss={avg_train:.6f} | Val Loss={avg_val:.6f}")
        
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            patience_counter = 0
            print("✅ Saved new best model.")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print("Early stopping triggered.")
                break
    return model

if __name__ == '__main__':
    # To run training, uncomment:
    train_model()